# Handwritten Digit Recognition — Perceptron vs. ANN vs. CNN

**Dataset:** MNIST (28×28 grayscale handwritten digits, 10 classes)

**Goal:** Build three progressively more powerful models on the same dataset and
compare them fairly — a single-layer Perceptron, a fully-connected ANN, and a
Convolutional Neural Network (CNN) — to see how model architecture affects
accuracy on image data.

**Notebook roadmap**
1. Imports & setup
2. Load and inspect the data
3. Preprocessing (normalization, reshaping, one-hot encoding)
4. Baseline model — Perceptron
5. Fully-connected Neural Network (ANN)
6. Convolutional Neural Network (CNN)
7. Training curves & model comparison
8. Error analysis (confusion matrix, misclassified digits, per-class metrics)
9. What the CNN actually "sees" (learned filters)
10. Final takeaways


## 1. Imports & Setup

In [ ]:
# --- Core data handling ---
import numpy as np
import pandas as pd

# --- Visualization ---
import seaborn as sns
import matplotlib.pyplot as plt

# A consistent, clean plotting style used throughout the notebook
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("Set2")
PALETTE = {"Perceptron": "#ef476f", "ANN": "#118ab2", "CNN": "#06d6a0"}

import warnings
warnings.filterwarnings('ignore')

# --- Preprocessing / evaluation utilities ---
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Perceptron            # simple linear classifier (baseline)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# --- Model building blocks (Keras / TensorFlow) ---
from tensorflow.keras.models import Sequential          # stack layers one after another
from tensorflow.keras.layers import Dense                # fully-connected layer, used for final predictions
from tensorflow.keras.layers import Conv2D                # learns spatial features (edges, curves, strokes)
from tensorflow.keras.layers import Flatten               # collapses a 2D feature map into a 1D vector
from tensorflow.keras.layers import MaxPooling2D           # downsamples feature maps, keeps strongest signals
from tensorflow.keras.layers import Dropout                 # randomly zeroes activations during training to fight overfitting
from tensorflow.keras.utils import to_categorical            # turns integer labels (0-9) into one-hot vectors


## 2. Load & Inspect the Data

MNIST here is provided as flat CSVs: each row is one image, the first column
is the digit label (0–9), and the remaining 784 columns are pixel intensities
(0–255) for the 28×28 image, flattened row-by-row.

In [ ]:
# Load train/test splits
df = pd.read_csv("mnist_train.csv")
df_test = pd.read_csv("mnist_test.csv")

print(f"Train rows: {df.shape[0]:,} | Test rows: {df_test.shape[0]:,} | Columns: {df.shape[1]}")
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
# Sanity check — make sure there are no missing pixel values
df.isnull().sum().sum()

### 2.1 What do the digits actually look like?

Before touching any model, it's worth looking at real samples — this is the
kind of visual check that tells you immediately whether the data loaded
correctly (right shape, right orientation, sensible label range).

In [ ]:
# Show a grid of random training digits with their labels
fig, axes = plt.subplots(3, 8, figsize=(14, 6))
rng = np.random.default_rng(42)
sample_idxs = rng.choice(len(df), size=24, replace=False)

for ax, idx in zip(axes.flat, sample_idxs):
    pixels = df.drop("label", axis=1).iloc[idx].values.reshape(28, 28)
    ax.imshow(pixels, cmap="gray")
    ax.set_title(str(df["label"].iloc[idx]), fontsize=11, fontweight="bold")
    ax.axis("off")

fig.suptitle("Random Sample of Training Digits", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 2.2 Are the classes balanced?

An uneven class distribution can quietly bias accuracy, so it's worth
checking before training anything.

In [ ]:
class_counts = df["label"].value_counts().sort_index()

plt.figure(figsize=(9, 5))
bars = plt.bar(class_counts.index, class_counts.values, color=sns.color_palette("crest", 10))
plt.title("Digit Class Distribution (Training Set)", fontsize=13, fontweight="bold")
plt.xlabel("Digit")
plt.ylabel("Number of Samples")
plt.xticks(range(10))
for bar, count in zip(bars, class_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
              f"{count:,}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

## 3. Preprocessing

Three steps before any model can use this data:
1. Split features (pixels) from labels
2. Scale pixel values from [0, 255] to [0, 1] — neural nets train faster and
   more stably on small, consistent input ranges
3. Reshape into the formats each model expects, and one-hot encode the labels
   for use with a softmax output layer

In [ ]:
# Split features/labels
X_train = df.drop("label", axis=1).values
y_train = df["label"].values
X_test = df_test.drop("label", axis=1).values
y_test = df_test["label"].values

In [ ]:
# Normalize pixel intensities to [0, 1]
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

In [ ]:
# Reshape flat 784-length vectors back into 28x28 images
# (Perceptron/ANN will flatten these again internally; CNN needs the 2D structure)
X_train_img = X_train.reshape(-1, 28, 28)
X_test_img = X_test.reshape(-1, 28, 28)

In [ ]:
# One-hot encode labels: digit 3 -> [0,0,0,1,0,0,0,0,0,0]
# Needed because the output layer uses softmax across 10 classes
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

## 4. Baseline Model — Single-Layer Perceptron

The simplest possible neural network: flatten the image into 784 numbers and
map them directly to 10 output classes with one layer. No hidden layers, no
non-linearity beyond softmax — this is our floor, not our goal.

In [ ]:
perceptron = Sequential([
    Flatten(input_shape=(28, 28)),      # 28x28 image -> 784-length vector
    Dense(10, activation="softmax")     # one weight per pixel per class
])

perceptron.summary()

In [ ]:
perceptron.compile(optimizer="sgd", loss="categorical_crossentropy", metrics=["accuracy"])

In [ ]:
history_percp = perceptron.fit(
    X_train_img, y_train_cat,
    epochs=5, batch_size=32,
    validation_data=(X_test_img, y_test_cat),
    verbose=1
)

In [ ]:
acc_percp = perceptron.evaluate(X_test_img, y_test_cat, verbose=0)[1]
print(f"Perceptron test accuracy: {acc_percp*100:.2f}%")

## 5. Fully-Connected Neural Network (ANN)

Adding two hidden layers (128 and 64 neurons, ReLU activation) lets the model
learn non-linear combinations of pixels instead of just a linear boundary per
class — the jump we'd expect to see from the Perceptron.

In [ ]:
# ANN with two hidden layers
ann = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(128, activation="relu"),      # hidden layer 1 — learns broad pixel patterns
    Dense(64, activation="relu"),       # hidden layer 2 — combines those into higher-level features
    Dense(10, activation="softmax")     # output layer — class probabilities
])

ann.summary()

In [ ]:
ann.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

In [ ]:
history_ann = ann.fit(
    X_train_img, y_train_cat,
    epochs=5, batch_size=32,
    validation_data=(X_test_img, y_test_cat),
    verbose=1
)

In [ ]:
acc_ann = ann.evaluate(X_test_img, y_test_cat, verbose=0)[1]
print(f"ANN test accuracy: {acc_ann*100:.2f}%")

## 6. Convolutional Neural Network (CNN)

Unlike the ANN, the CNN keeps the image's 2D spatial structure intact. Conv2D
layers learn local patterns (edges, curves, loops) with shared filters, and
MaxPooling progressively shrinks the feature maps while keeping the strongest
signals — this is the architecture built specifically for image data.

In [ ]:
# CNN needs an explicit channel dimension: (28, 28) -> (28, 28, 1)
X_train_cnn = X_train.reshape(-1, 28, 28, 1)
X_test_cnn = X_test.reshape(-1, 28, 28, 1)

In [ ]:
cnn = Sequential([
    Conv2D(32, kernel_size=(3, 3), activation="relu", input_shape=(28, 28, 1)),  # learns 32 low-level filters (edges/strokes)
    MaxPooling2D(pool_size=(2, 2)),                                              # downsample, keep strongest activations
    Conv2D(64, kernel_size=(3, 3), activation="relu"),                           # learns 64 higher-level filters (shapes/loops)
    MaxPooling2D(pool_size=(2, 2)),
    Flatten(),                                                                   # collapse feature maps into a vector
    Dense(128, activation="relu"),                                              # combine features for classification
    Dropout(0.5),                                                                # randomly drop half the units to reduce overfitting
    Dense(10, activation="softmax")                                             # final class probabilities
])

cnn.summary()

In [ ]:
cnn.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

In [ ]:
history_cnn = cnn.fit(
    X_train_cnn, y_train_cat,
    epochs=5, batch_size=32,
    validation_data=(X_test_cnn, y_test_cat),
    verbose=1
)

In [ ]:
acc_cnn = cnn.evaluate(X_test_cnn, y_test_cat, verbose=0)[1]
print(f"CNN test accuracy: {acc_cnn*100:.2f}%")

## 7. Training Curves & Model Comparison

In [ ]:
def plot_training(history, title, color):
    """Plot train/val accuracy and loss curves side by side for one model."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history.history['accuracy'], label="Train", color=color, linewidth=2)
    axes[0].plot(history.history['val_accuracy'], label="Val", color=color, linestyle="--", linewidth=2)
    axes[0].set_title(f"{title} — Accuracy", fontweight="bold")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(history.history['loss'], label="Train", color=color, linewidth=2)
    axes[1].plot(history.history['val_loss'], label="Val", color=color, linestyle="--", linewidth=2)
    axes[1].set_title(f"{title} — Loss", fontweight="bold")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
plot_training(history_percp, "Perceptron", PALETTE["Perceptron"])

In [ ]:
plot_training(history_ann, "ANN", PALETTE["ANN"])

In [ ]:
plot_training(history_cnn, "CNN", PALETTE["CNN"])

### 7.1 Head-to-head validation accuracy

Putting all three on one chart makes the architecture effect obvious at a
glance — each step (linear → non-linear → convolutional) should show a clear
lift, with diminishing improvements as the models get harder to train.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(history_percp.history['val_accuracy'], label="Perceptron", color=PALETTE["Perceptron"], marker="o", linewidth=2)
plt.plot(history_ann.history['val_accuracy'], label="ANN", color=PALETTE["ANN"], marker="o", linewidth=2)
plt.plot(history_cnn.history['val_accuracy'], label="CNN", color=PALETTE["CNN"], marker="o", linewidth=2)
plt.title("Validation Accuracy — Perceptron vs. ANN vs. CNN", fontsize=13, fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
final_accs = [acc_percp * 100, acc_ann * 100, acc_cnn * 100]
models = ["Perceptron", "ANN", "CNN"]

plt.figure(figsize=(8, 6))
bars = plt.bar(models, final_accs, color=[PALETTE[m] for m in models])
plt.title("Final Test Accuracy Comparison", fontsize=13, fontweight="bold")
plt.ylabel("Accuracy (%)")
for bar, acc in zip(bars, final_accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 1, f"{acc:.2f}%",
              ha='center', va='bottom', fontsize=12, fontweight='bold', color="white")
plt.ylim(80, 100)
plt.tight_layout()
plt.show()

## 8. Error Analysis

Overall accuracy hides *where* a model struggles. This section looks at the
CNN's mistakes specifically, since it's the strongest model: which digits get
confused with which, and what the actual misclassified images look like.

In [ ]:
def show_side_by_side(models, model_names, X, X_cnn, y_true, n=5, seed=0):
    """Show the same n random test digits and each model's prediction for them."""
    rng = np.random.default_rng(seed)
    idxs = rng.choice(len(X), n, replace=False)
    plt.figure(figsize=(15, 3.5 * len(models)))

    for row, (model, name) in enumerate(zip(models, model_names)):
        for i, idx in enumerate(idxs):
            plt.subplot(len(models), n, row * n + i + 1)
            plt.imshow(X[idx].reshape(28, 28), cmap="gray")
            plt.axis("off")
            inp = X_cnn[idx].reshape(1, 28, 28, 1) if name == "CNN" else X[idx].reshape(1, 28, 28)
            pred = np.argmax(model.predict(inp, verbose=0))
            correct = pred == y_true[idx]
            color = "green" if correct else "red"
            plt.title(f"{name}: {pred} (true {y_true[idx]})", color=color, fontsize=9)

    plt.tight_layout()
    plt.show()

In [ ]:
show_side_by_side([perceptron, ann, cnn], ["Perceptron", "ANN", "CNN"], X_test_img, X_test_cnn, y_test, n=5)

### 8.1 Confusion matrix (CNN)

Which digits does the best model still mix up? Diagonal = correct
predictions; anything off the diagonal is a specific pair of digits the
model confuses (classically 4↔9 and 3↔5↔8 for MNIST).

In [ ]:
y_pred_cnn = np.argmax(cnn.predict(X_test_cnn, verbose=0), axis=1)
cm = confusion_matrix(y_test, y_pred_cnn)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=True)
plt.title("CNN Confusion Matrix", fontsize=13, fontweight="bold")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()

### 8.2 Actual misclassified digits

Numbers in a confusion matrix are abstract — seeing the actual images the
CNN got wrong makes it clear that many "errors" are genuinely ambiguous
handwriting.

In [ ]:
wrong_idxs = np.where(y_pred_cnn != y_test)[0]
n_show = min(15, len(wrong_idxs))
sample_wrong = np.random.default_rng(1).choice(wrong_idxs, size=n_show, replace=False)

fig, axes = plt.subplots(3, 5, figsize=(13, 8))
for ax, idx in zip(axes.flat, sample_wrong):
    ax.imshow(X_test_cnn[idx].reshape(28, 28), cmap="gray")
    ax.set_title(f"True: {y_test[idx]} | Pred: {y_pred_cnn[idx]}", color="crimson", fontsize=10)
    ax.axis("off")

fig.suptitle("CNN Misclassified Digits", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 8.3 Per-class precision / recall / F1 (CNN)

A single accuracy number can't tell you if the model is quietly weaker on
one digit than the rest — this breaks performance down class by class.

In [ ]:
report = classification_report(y_test, y_pred_cnn, output_dict=True)
report_df = pd.DataFrame(report).T.iloc[:10][["precision", "recall", "f1-score"]]

report_df.plot(kind="bar", figsize=(11, 5), color=["#ef476f", "#118ab2", "#06d6a0"])
plt.title("CNN Per-Class Precision / Recall / F1", fontsize=13, fontweight="bold")
plt.xlabel("Digit")
plt.ylabel("Score")
plt.ylim(0.9, 1.01)
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 9. What Is the CNN Actually Looking At?

CNNs are often treated as black boxes, but the first convolutional layer's
learned filters are easy to inspect directly — they're small (3×3) edge/
stroke detectors applied across the whole image.

In [ ]:
# Grab the weights of the first Conv2D layer: shape (3, 3, 1, 32)
first_conv_weights = cnn.layers[0].get_weights()[0]

fig, axes = plt.subplots(4, 8, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    ax.imshow(first_conv_weights[:, :, 0, i], cmap="viridis")
    ax.axis("off")

fig.suptitle("First Conv2D Layer — 32 Learned 3×3 Filters", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 9.1 Feature maps for a single digit

Passing one real digit through the first conv layer shows what each filter
activates on — some pick up edges, others curves or strokes in specific
orientations.

In [ ]:
from tensorflow.keras.models import Model

# Build a model that outputs the activations of just the first Conv2D layer
activation_model = Model(inputs=cnn.inputs, outputs=cnn.layers[0].output)
sample_digit = X_test_cnn[0:1]
feature_maps = activation_model.predict(sample_digit, verbose=0)[0]  # shape (26, 26, 32)

fig, axes = plt.subplots(4, 8, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    ax.imshow(feature_maps[:, :, i], cmap="viridis")
    ax.axis("off")

fig.suptitle(f"Feature Maps for a Single '{y_test[0]}' Digit (First Conv Layer)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 10. Takeaways

- **Perceptron** — a single linear layer is a reasonable floor, but with no
  hidden layers it can't capture the non-linear patterns that separate
  visually similar digits.
- **ANN** — two dense hidden layers close a meaningful chunk of that gap by
  learning non-linear pixel combinations, but still treats the image as a
  flat vector, discarding spatial structure.
- **CNN** — convolution + pooling exploits the fact that nearby pixels are
  related, learning reusable local features (edges, strokes, loops). This is
  why it comes out on top, and the confusion matrix shows its remaining
  errors are concentrated in genuinely ambiguous digit pairs (e.g. 4/9,
  3/5/8) rather than being spread randomly.
